# DCGAN on WikiArt: run everything

The code lives in the `.py` files next to this notebook (`data.py`, `models.py`, `train.py`, `evaluate.py`, `plots.py`).

**Setup**
1. Upload the `gan_wikiart` folder to Google Drive as `MyDrive/gan_wikiart_code` (or change `CODE_DIR` below).
2. Add a Colab secret `KAGGLE_API_TOKEN` (kaggle.com/settings/api → *Generate New Token*) and enable notebook access.
3. *Runtime → Change runtime type → T4 GPU*, then *Run all*.

The resized dataset is cached on Drive, so the ~30 GB download happens only once (an existing cache from the old notebook is reused).
Training checkpoints every epoch; if Colab disconnects, *Run all* again and it resumes.

In [ ]:
%pip install -q -U pytorch-ignite kagglehub
from google.colab import drive
drive.mount("/content/drive")

import sys
CODE_DIR = "/content/drive/MyDrive/gan_wikiart_code"   # folder with the .py files
sys.path.insert(0, CODE_DIR)

In [ ]:
import json
from pathlib import Path
import torch
from IPython.display import Image, display

device = "cuda"
OUT = Path("/content/drive/MyDrive/gan_wikiart/run")
CACHE = "/content/drive/MyDrive/gan_wikiart/data_cache/kaggle_64px_all_full.npz"
OUT.mkdir(parents=True, exist_ok=True)

CFG = dict(epochs=50, batch_size=128, lr=2e-4, beta1=0.5, nz=100, real_label=0.9, eval_every=5)
EVAL_N, FINAL_EVAL_N, SEEDS = 5000, 10000, 3   # FID needs > 2048 images
json.dump(CFG, open(OUT / "config.json", "w"), indent=2)
torch.manual_seed(42)

## Data

In [ ]:
from data import load_wikiart
from plots import save_grid

X, styles = load_wikiart(CACHE)
X = X.to(device)   # whole dataset on the GPU as uint8 (~1 GB)
print(X.shape, "|", len(set(styles)), "styles")
perm = torch.randperm(len(X), device=device)   # fixed real subsets for evaluation
save_grid(X[perm[:64]].float() / 127.5 - 1, OUT / "real_samples.png")
display(Image(OUT / "real_samples.png"))

## Train

In [ ]:
from models import Generator, Discriminator, weights_init
from train import train

G = Generator(CFG["nz"]).to(device).apply(weights_init)
D = Discriminator().to(device).apply(weights_init)
logs = train(G, D, X, OUT, eval_idx=perm[:EVAL_N], **CFG)

In [ ]:
from plots import plot_training
plot_training(logs, OUT / "training.pdf")

## Final FID / IS

In [ ]:
import numpy as np
from evaluate import evaluate_generator, evaluate_real

real_a, real_b = perm[:FINAL_EVAL_N], perm[FINAL_EVAL_N:2 * FINAL_EVAL_N]
runs = np.array([evaluate_generator(G, X, real_a, CFG["nz"], seed=1000 + s) for s in range(SEEDS)])
fid_real, is_real = evaluate_real(X, real_a, real_b)

metrics = dict(n=FINAL_EVAL_N, fid_mean=runs[:, 0].mean(), fid_std=runs[:, 0].std(),
               is_mean=runs[:, 1].mean(), is_std=runs[:, 1].std(),
               fid_real_vs_real=fid_real, is_real=is_real,
               train_minutes=sum(e["seconds"] for e in logs["epochs"]) / 60)
metrics = {k: float(v) for k, v in metrics.items()} | {"n": FINAL_EVAL_N}
json.dump(metrics, open(OUT / "final_metrics.json", "w"), indent=2)
print(json.dumps(metrics, indent=2))

## Generated images

In [ ]:
from plots import nearest_neighbors

G.eval()
with torch.no_grad():
    samples = G(torch.randn(64, CFG["nz"], 1, 1, device=device))
save_grid(samples, OUT / "generated.png")
nearest_neighbors(samples[:6], X, OUT / "nearest_neighbors.png")
display(Image(OUT / "generated.png"), Image(OUT / "nearest_neighbors.png"))

In [ ]:
# zip everything except the checkpoint and download it
import shutil
from google.colab import files
tmp = Path("/content/results"); shutil.rmtree(tmp, ignore_errors=True)
shutil.copytree(OUT, tmp, ignore=shutil.ignore_patterns("checkpoint.pt"))
files.download(shutil.make_archive("/content/gan_wikiart_results", "zip", tmp))